# Final models: inference on the focal-sampling data, TabPFN

Runs the TabPFN B2 uncorrected model from `10a_final_models_training_tabpfn` on the focal-sampled B2 uncorrected features, with the same day-by-day reading, column types and output format as the RF in `10b_final_models_inference`. It is a comparison model in the focal-sampling comparison (`11a`, `12d`); the final classifier, HR B2 rotdaily, is run in `10b_final_models_inference`. TabPFN is in its own notebook because it needs the `tabpfn` environment.

Output (`data/output/inference_results/`): `final_tabpfn_b2_unc_focal_sampled_random_predictions.parquet`, TabPFN label and class probabilities per burst.

In [ ]:
import datetime
import os
import warnings

import joblib
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pytz
from tqdm import tqdm

In [ ]:
TAB_DIR, TAB_NAME = "../models/final_tabpfn_b2_unc", "final_tabpfn_b2_unc"
FOCAL_FEATURES = (
    "../data/raw/focal_sampling/features/focal_sampled_features_burst_2_uncorrected.parquet"
)
OUT_DIR = "../data/output/inference_results"
TAB_OUT = f"{OUT_DIR}/{TAB_NAME}_focal_sampled_random_predictions.parquet"
# Rows per predict_proba call, to bound GPU memory
CHUNK_SIZE = 10000

CLASS_COLS = [
    "Eating",
    "Grooming actor",
    "Grooming receiver",
    "Resting",
    "Running",
    "Self-scratching",
    "Sleeping",
    "Walking",
]
ID_COLS = ["Ind_ID", "burst_id", "new_burst"]
TZ = pytz.timezone("Africa/Johannesburg")

# Schema for the per-model prediction files
parquet_schema = pa.schema(
    [
        ("Ind_ID", pa.string()),
        ("burst_id", pa.int32()),
        ("new_burst", pa.int32()),
        ("burst_start_time", pa.timestamp("ns", tz="Africa/Johannesburg")),
        ("predicted_behaviour", pa.string()),
    ]
    + [(c, pa.float32()) for c in CLASS_COLS]
)


def day_bounds(date):
    """Timezone-aware start and end of a day, for parquet row filters."""
    start_dt = TZ.localize(datetime.datetime.combine(date, datetime.time(0, 0, 0)))
    end_dt = TZ.localize(datetime.datetime.combine(date, datetime.time(23, 59, 59)))
    return start_dt, end_dt


def unique_dates(pq_file_path):
    """Sorted unique dates of burst_start_time in a parquet file."""
    df = pd.read_parquet(pq_file_path, columns=["burst_start_time"])
    dates = sorted(pd.to_datetime(df["burst_start_time"]).dt.date.unique())
    print(f"Found {len(dates)} unique dates in {os.path.basename(pq_file_path)}")
    return dates


def format_and_save(final_df, class_names, output_parquet):
    """Fix column types and order, and save a per-model prediction file."""
    final_df["Ind_ID"] = final_df["Ind_ID"].astype(str)
    final_df["burst_id"] = final_df["burst_id"].astype(int)
    final_df["new_burst"] = final_df["new_burst"].astype(int)
    final_df["predicted_behaviour"] = final_df["predicted_behaviour"].astype(str)
    assert list(class_names) == CLASS_COLS, class_names
    for col in CLASS_COLS:
        final_df[col] = final_df[col].astype(float).round(4)
    final_df = final_df[ID_COLS + ["burst_start_time", "predicted_behaviour"] + CLASS_COLS]
    table = pa.Table.from_pandas(final_df, schema=parquet_schema, preserve_index=False)
    pq.write_table(table, output_parquet, compression="zstd", compression_level=9)
    print(f"Saved {len(final_df)} predictions to {output_parquet}")
    return final_df

In [ ]:
tab = joblib.load(f"{TAB_DIR}/{TAB_NAME}_tabpfn_model.joblib")
tab_encoder = joblib.load(f"{TAB_DIR}/{TAB_NAME}_label_encoder.joblib")
tab_classes = list(tab_encoder.classes_)
# Feature order the model was trained with
tab_features = list(tab.feature_names_in_)

tab_list = []
for date in tqdm(unique_dates(FOCAL_FEATURES), desc="TabPFN: processing dates"):
    start_dt, end_dt = day_bounds(date)
    df = pq.read_table(
        FOCAL_FEATURES,
        filters=[
            ("burst_start_time", ">=", start_dt),
            ("burst_start_time", "<", end_dt),
        ],
    ).to_pandas()
    if df.empty:
        continue
    X = df[tab_features]
    n_nan = int(X.isna().any(axis=1).sum())
    if n_nan:
        print(f"WARNING: {n_nan} bursts with missing features on {date}")
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        proba = np.vstack(
            [
                tab.predict_proba(X.iloc[i : i + CHUNK_SIZE])
                for i in range(0, len(X), CHUNK_SIZE)
            ]
        )
    probas = pd.DataFrame(proba, columns=tab_classes)
    probas["predicted_behaviour"] = probas[tab_classes].idxmax(axis=1)
    tab_list.append(
        pd.concat(
            [df[ID_COLS + ["burst_start_time"]].reset_index(drop=True), probas], axis=1
        )
    )

tab_pred = format_and_save(pd.concat(tab_list, ignore_index=True), tab_classes, TAB_OUT)